In [ ]:
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np

# the path of the images to be analyzed
l_path = "l_183x249.png"
r_path = "r_183x249.png"
paths = [l_path, r_path]
i_label = ["L", "R"]

# load images and show
print()
print(f"(A)")
imgs = [mpimg.imread(x) for x in paths]
H, W, _ = imgs[0].shape
N = H * W

fig, axes = plt.subplots(1, len(imgs))
for i, S in enumerate(imgs):
    axes[i].imshow(S)
    axes[i].set_title(f"$\\hat S_{i_label[i]}$")
    axes[i].set_xlabel(f"$x$")
    axes[i].set_ylabel(f"$y$")
    
fig.subplots_adjust(wspace=0.5)
plt.show()

# (I)
hat_S_values = np.array([255])
#hat_S_values = np.array([127, 63, 31, 15, 7, 3, 1])

for hat_S in hat_S_values:
    print()
    print(f"(I)")
    print(f"$\\hat S$ = {hat_S}")

    # prepare images
    print()
    print(f"(B)")
    for i in range(len(imgs)):
        S = imgs[i]

        if S.ndim == 3:
            S = S.mean(axis=2)

        normalized = (S - S.min()) / (S.max() - S.min()) * (hat_S - 1)
        normalized = normalized.astype(np.uint8)

        imgs[i] = normalized

    # plot prepared images
    fig, axes = plt.subplots(1, len(imgs))
    for i, S in enumerate(imgs):
        axes[i].imshow(S, cmap="gray")
        axes[i].set_title(f"$S_{i_label[i]}$")
        axes[i].set_xlabel(f"$x$")
        axes[i].set_ylabel(f"$y$")
        
    fig.subplots_adjust(wspace=0.5)
    plt.show()

    # histogram
    hist = [np.bincount(x.ravel(), minlength=hat_S) for x in imgs]

    # calculate probabilities
    print()
    print(f"(C)")
    Probability = [hist[i]/N for i, x in enumerate(imgs)]

    for i, p in enumerate(Probability):
        plt.bar(np.arange(hat_S), p, width=1, alpha=0.5, label=f"$S_{i_label[i]}$")
        
    plt.xlabel(f"$S_i\\left(x, y\\right)$")
    plt.ylabel(f"Probability $P\\left(S_i\\right)$")
    plt.legend()
    plt.show()

    # sanity check
    for i, p in enumerate(Probability):
        total = np.sum(p)
        print(f"sanity check {i}: {total}") # should print 1.0

    # pixel entropy
    print()
    print(f"(D)")
    PixelEntropy = [- np.sum(p * np.log2(p)) for p in Probability]

    print(f"pixel entropy:")
    for i, h in enumerate(PixelEntropy):
        print(f"$H(S_{i_label[i]}) = {h}$")
        
    # joint probability
    print()
    print(f"(E)")

    JointProbability = np.zeros((hat_S, hat_S))
    for y in range(H):
        for x in range(W):
            s0 = imgs[0][y, x]
            s1 = imgs[1][y, x]
            JointProbability[s0, s1] += 1

    JointProbability = JointProbability / JointProbability.sum()

    plot = plt.imshow(JointProbability)
    plt.colorbar(plot)
    plt.title("Joint probability $P(S_1, S_2)$")
    plt.xlabel("$S_1$")
    plt.ylabel("$S_2$")
    plt.show()

    total = np.sum(JointProbability)
    print(f"sanity check: {total}") # should print 1.0

    # joint entropy
    print()
    print(f"(F)")
    JointEntropy = 0

    for S_1 in range(hat_S):
        for S_2 in range(hat_S):
            if JointProbability[S_1, S_2] == 0.0:
                continue

            JointEntropy -= JointProbability[S_1, S_2] * np.log2(JointProbability[S_1, S_2])

    print(f"joint entropy:")
    print(f"$H(S_1,S_2)={JointEntropy}$")

    # mutual information
    print()
    print(f"(G)")
    MutualInformationProbability = 0

    for S_1 in range(hat_S):
        for S_2 in range(hat_S):
            PS_1 = Probability[0][S_1]
            PS_2 = Probability[1][S_2]

            if PS_1 * PS_2 == 0.0:
                continue
            if JointProbability[S_1, S_2] == 0.0:
                continue

            MutualInformationProbability += JointProbability[S_1, S_2] * np.log2(JointProbability[S_1, S_2] / (PS_1 * PS_2))

    MutualInformationEntropy = PixelEntropy[0] + PixelEntropy[1] - JointEntropy

    print(f"mutual information:")
    print(f"$I(S_1,S_2)={MutualInformationProbability}$")
    print(f"$I(S_1,S_2)={MutualInformationEntropy}$")

    # redundancy
    print()
    print(f"(H)")

    Redundancy = (PixelEntropy[0] + PixelEntropy[1]) / JointEntropy - 1
    print(f"Redundancy = {Redundancy}")

    # correlation
    print()
    print(f"(J)")

    Average = [x.sum() / N for x in imgs]
    S = [imgs[i] - Average[i] for i in range(len(imgs))]

    R = np.zeros((2,2))
    for j in range(2):
        for i in range(2):

            for y in range(H):
                for x in range(W):
                    S_i = S[i][y, x]
                    S_j = S[j][y, x]
                    R[j, i] += float(S_i) * float(S_j)
            
            R[j, i] /= N
    
    print(f"R =")
    print(f"{R}")

    # scatter
    print()
    print(f"(K)")

    plt.scatter(S[0], S[1], s=1)
    plt.xlabel("$S_L$")
    plt.ylabel("$S_R$")
    plt.show()

    plt.show()


    # eigenvalues and eigenvectors
    print()
    print(f"(L)")

    eigenvalues, eigenvectors = np.linalg.eig(R)
    scales = np.sqrt(eigenvalues)

    x = S[0].flatten()
    y = S[1].flatten()
    plt.scatter(x, y, s=1)

    mean_x = np.mean(x)
    mean_y = np.mean(y)
    for i in range(2):
        #scale = np.sqrt(eigenvalues[i])
        vec = eigenvectors[:, i] * scales[i]
        plt.arrow(
            mean_x, mean_y,
            vec[0], vec[1],
            head_width=scales.min() / 5,
            color='red'
        )

    plt.xlabel("$S_L$")
    plt.ylabel("$S_R$")
    plt.show()

    # projections
    print()
    print(f"(M)")
    S_pos = (S[0]+S[1])/np.sqrt(2)
    S_neg = (-S[0]+S[1])/np.sqrt(2)

    fig, axes = plt.subplots(1, len(S))

    axes[0].imshow(S_pos, cmap="gray")
    axes[0].set_title(f"$S_{{+}}$")
    axes[0].set_xlabel(f"$x$")
    axes[0].set_ylabel(f"$y$")

    axes[1].imshow(S_neg, cmap="gray")
    axes[1].set_title(f"$S_{{-}}$")
    axes[1].set_xlabel(f"$x$")
    axes[1].set_ylabel(f"$y$")
        
    fig.subplots_adjust(wspace=0.5)
    plt.show()

    # correlation matrix
    print()
    print(f"(N)")

    S_proj = np.array([S_pos, S_neg])

    R = np.zeros((2,2))
    for j in range(2):
        for i in range(2):

            for y in range(H):
                for x in range(W):
                    S_i = S_proj[i][y, x]
                    S_j = S_proj[j][y, x]
                    R[j, i] += float(S_i) * float(S_j)
            
            R[j, i] /= N
    
    print(f"R =")
    print(f"{R}")

    eigenvalues, eigenvectors = np.linalg.eig(R)
    scales = np.sqrt(eigenvalues)

    x = S_proj[0].flatten()
    y = S_proj[1].flatten()
    plt.scatter(x, y, s=1)

    mean_x = np.mean(x)
    mean_y = np.mean(y)
    for i in range(2):
        vec = eigenvectors[:, i] * scales[i]
        plt.arrow(
            mean_x, mean_y,
            vec[0], vec[1],
            head_width=scales.min() / 5,
            color='red'
        )

    plt.xlabel(f"$S_{{+}}$")
    plt.ylabel(f"$S_{{-}}$")
    plt.show()

    if R[0,0] > R[1,1]:
        larger_message = f"$S_{{+}}$"
    else:
        larger_message = f"$S_{{-}}$"

    print(f"variance($S_{{+}}$) = {R[0,0]} and\nvariance($S_{{-}}$) = {R[1,1]},\ntherefore the variance of {larger_message} is larger.")

    # gain control
    print()
    print(f"(O)")

    g_pos = 1/np.sqrt(R[0,0])
    g_neg = 1/np.sqrt(R[1,1])

    O_pos = g_pos*S_pos
    O_neg = g_neg*S_neg

    fig, axes = plt.subplots(1, 2)
    axes[0].imshow(O_pos, cmap="gray")
    axes[0].set_title(f"$\\mathcal{{O}}_{{+}}$")
    axes[0].set_xlabel(f"$x$")
    axes[0].set_ylabel(f"$y$")
    axes[1].imshow(O_neg, cmap="gray")
    axes[1].set_title(f"$\\mathcal{{O}}_{{-}}$")
    axes[1].set_xlabel(f"$x$")
    axes[1].set_ylabel(f"$y$")
        
    fig.subplots_adjust(wspace=0.5)
    plt.show()

    # scatter after gain
    print()
    print(f"(P)")

    O = np.array([O_pos, O_neg])

    R = np.zeros((2,2))
    for j in range(2):
        for i in range(2):

            for y in range(H):
                for x in range(W):
                    O_i = O[i][y, x]
                    O_j = O[j][y, x]
                    R[j, i] += float(O_i) * float(O_j)
            
            R[j, i] /= N
    
    print(f"R =")
    print(f"{R}")

    eigenvalues, eigenvectors = np.linalg.eig(R)
    scales = np.sqrt(eigenvalues)

    x = O[0].flatten()
    y = O[1].flatten()
    plt.scatter(x, y, s=1)

    mean_x = np.mean(x)
    mean_y = np.mean(y)
    for i in range(2):
        vec = eigenvectors[:, i] * scales[i]
        plt.arrow(
            mean_x, mean_y,
            vec[0], vec[1],
            head_width=scales.min()/5,
            color='red'
        )

    plt.xlabel(f"$\\mathcal{{O}}_{{+}}$")
    plt.ylabel(f"$\\mathcal{{O}}_{{-}}$")
    plt.show()

    # reconstruct
    print()
    print(f"(Q)")

    O_1 = (O_pos + O_neg)/np.sqrt(2)
    O_2 = (O_pos - O_neg)/np.sqrt(2)
    
    fig, axes = plt.subplots(1, 2)
    axes[0].imshow(O_1, cmap="gray")
    axes[0].set_title(f"$\\mathcal{{O}}_{{1}}$")
    axes[0].set_xlabel(f"$x$")
    axes[0].set_ylabel(f"$y$")
    axes[1].imshow(O_2, cmap="gray")
    axes[1].set_title(f"$\\mathcal{{O}}_{{2}}$")
    axes[1].set_xlabel(f"$x$")
    axes[1].set_ylabel(f"$y$")
        
    fig.subplots_adjust(wspace=0.5)
    plt.show()

    # scatter the reconstruction
    print()
    print(f"(R)")

    O = np.array([O_1, O_2])
    R = np.zeros((2,2))
    for j in range(2):
        for i in range(2):

            for y in range(H):
                for x in range(W):
                    O_i = O[i][y, x]
                    O_j = O[j][y, x]
                    R[j, i] += float(O_i) * float(O_j)
            
            R[j, i] /= N
    
    print(f"R =")
    print(f"{R}")

    eigenvalues, eigenvectors = np.linalg.eig(R)
    scales = np.sqrt(eigenvalues)

    x = O[0].flatten()
    y = O[1].flatten()
    plt.scatter(x, y, s=1)

    mean_x = np.mean(x)
    mean_y = np.mean(y)
    for i in range(2):
        vec = eigenvectors[:, i] * scales[i]
        plt.arrow(
            mean_x, mean_y,
            vec[0], vec[1],
            head_width=scales.min()/5,
            color='red'
        )

    plt.xlabel(f"$\\mathcal{{O}}_{{1}}$")
    plt.ylabel(f"$\\mathcal{{O}}_{{2}}$")
    plt.show()